# 📘 IoT Building Blocks – Hands-on Lab (Jupyter Notebook)

## 🧪 Lab Overview
Build a complete IoT pipeline:

**Sensor → MQTT → Edge Processing → Database → Visualization → Action**

> **Prerequisites (one-time outside Jupyter):**
> - Install **Mosquitto MQTT Broker**: https://mosquitto.org/download/
> - Install **MongoDB Community Server**: https://www.mongodb.com/try/download/community
> - Start Mosquitto broker (e.g., `mosquitto` in terminal)
> - Ensure MongoDB is running (`mongod`)

## 🔹 Step 0: Setup Environment

In [1]:
# !pip install paho-mqtt pymongo pandas matplotlib

## 🔹 Step 1: Simulate IoT Device (Sensor Layer)

In [2]:
import random
import time
import json

def generate_sensor_data():
    data = {
        "temperature": round(random.uniform(20, 35), 2),
        "humidity": round(random.uniform(40, 70), 2),
        "device_id": "sensor_001",
        "timestamp": time.time()
    }
    return data

# Test
print(generate_sensor_data())

{'temperature': 25.79, 'humidity': 49.21, 'device_id': 'sensor_001', 'timestamp': 1783157620.4920728}


## 🔹 Step 2: MQTT Communication (Connectivity Layer) – Publisher

Make sure Mosquitto is running locally on port 1883.

In [3]:
import paho.mqtt.client as mqtt
import time, json

broker = "localhost"
port = 1883
topic = "iot/sensor/data"

client = mqtt.Client()
client.connect(broker, port)

print("Publishing sensor data... Press Ctrl+C to stop.")
try:
    while True:
        data = generate_sensor_data()
        client.publish(topic, json.dumps(data))
        print("Published:", data)
        time.sleep(2)
except KeyboardInterrupt:
    print("Stopped publisher.")
    client.disconnect()

/tmp/ipykernel_40506/3090296799.py:8: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


Publishing sensor data... Press Ctrl+C to stop.
Published: {'temperature': 32.06, 'humidity': 42.94, 'device_id': 'sensor_001', 'timestamp': 1783158440.1311345}
Published: {'temperature': 28.24, 'humidity': 66.94, 'device_id': 'sensor_001', 'timestamp': 1783158442.1314175}
Published: {'temperature': 26.63, 'humidity': 57.76, 'device_id': 'sensor_001', 'timestamp': 1783158444.1319106}
Published: {'temperature': 26.17, 'humidity': 57.58, 'device_id': 'sensor_001', 'timestamp': 1783158446.1331022}
Published: {'temperature': 26.67, 'humidity': 40.03, 'device_id': 'sensor_001', 'timestamp': 1783158448.1344278}
Published: {'temperature': 32.16, 'humidity': 63.94, 'device_id': 'sensor_001', 'timestamp': 1783158450.1349344}
Published: {'temperature': 24.19, 'humidity': 64.73, 'device_id': 'sensor_001', 'timestamp': 1783158452.1354895}
Published: {'temperature': 24.82, 'humidity': 50.94, 'device_id': 'sensor_001', 'timestamp': 1783158454.1359913}
Published: {'temperature': 27.26, 'humidity': 46

## 🔹 Step 3: MQTT Subscriber (Edge Layer Processing)

In [ ]:
import paho.mqtt.client as mqtt
import json

def on_message(client, userdata, msg):
    data = json.loads(msg.payload.decode())
    print("Received:", data)

    # Edge Processing Logic
    if data["temperature"] > 30:
        print("⚠️ ALERT: High Temperature!")

client = mqtt.Client()
client.connect("localhost", 1883)

client.subscribe("iot/sensor/data")
client.on_message = on_message

print("Listening for messages... Press Ctrl+C to stop.")
try:
    client.loop_forever()
except KeyboardInterrupt:
    print("Stopped subscriber.")
    client.disconnect()

## 🔹 Step 4: Store Data in MongoDB (Cloud Layer)

In [ ]:
from pymongo import MongoClient
import paho.mqtt.client as mqtt
import json

mongo_client = MongoClient("mongodb://localhost:27017/")
db = mongo_client["iot_db"]
collection = db["sensor_data"]

def on_message_store(client, userdata, msg):
    data = json.loads(msg.payload.decode())
    collection.insert_one(data)
    print("Stored in DB:", data)

client = mqtt.Client()
client.connect("localhost", 1883)

client.subscribe("iot/sensor/data")
client.on_message = on_message_store

print("Storing incoming data to MongoDB... Press Ctrl+C to stop.")
try:
    client.loop_forever()
except KeyboardInterrupt:
    print("Stopped DB subscriber.")
    client.disconnect()

## 🔹 Step 5: Data Visualization (Application Layer)

In [ ]:
import pandas as pd

data = list(collection.find())
df = pd.DataFrame(data)

df[['temperature', 'humidity']].tail(10)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(df['temperature'], label='Temperature')
plt.plot(df['humidity'], label='Humidity')
plt.legend()
plt.title("Sensor Data Trends")
plt.xlabel("Index")
plt.ylabel("Value")
plt.show()

## 🔹 Step 6: Simulate Actuator (Physical Action Layer)

In [ ]:
def actuator_control(data):
    if data["temperature"] > 30:
        print("🌀 Fan ON")
    else:
        print("❌ Fan OFF")

# Test
actuator_control(generate_sensor_data())

## 🔹 Step 7: Full Pipeline Integration

In [ ]:
import paho.mqtt.client as mqtt
import json

def on_message_full(client, userdata, msg):
    data = json.loads(msg.payload.decode())

    # Store
    collection.insert_one(data)

    # Decision
    actuator_control(data)

    print("Processed:", data)

client = mqtt.Client()
client.connect("localhost", 1883)

client.subscribe("iot/sensor/data")
client.on_message = on_message_full

print("Running full pipeline... Press Ctrl+C to stop.")
try:
    client.loop_forever()
except KeyboardInterrupt:
    print("Stopped full pipeline.")
    client.disconnect()

## 🔹 Step 8: Architecture Mapping

| Component | Layer |
|----------|------|
| Sensor simulation | Device |
| MQTT | Connectivity |
| Python processing | Edge |
| MongoDB | Cloud |
| Visualization | Application |
| Actuator logic | Physical |

## 🔹 Step 9: Mini Challenge

- Add humidity alert  
- Store only filtered data  
- Add device ID-based filtering

## 🔹 Step 10: Key Takeaway

You built:
- Device (sensor simulation)
- Connectivity (MQTT)
- Edge processing (Python logic)
- Cloud storage (MongoDB)
- Application (Visualization)
- Action (Actuator simulation)

👉 This is a **complete IoT system in practice**